# Expanded Exercise: Multiple Group Comparisons (VeryAnts Sales)
## One-way ANOVA + Pairwise Post-hoc Tests with Correction, Effect Sizes & Simulation

**Learning Goals:**
- Understand why running multiple uncorrected t-tests inflates Type I error (family-wise error rate)
- Perform and interpret one-way ANOVA as the omnibus test
- Apply post-hoc pairwise comparisons with correction (Bonferroni, Tukey HSD)
- Check assumptions and compute effect sizes + CIs
- Use simulation to explore false positive rates and power
- Report results in an audience-aware way following data analysis report structure

This exercise expands the original Codecademy-style multiple t-tests prompt into a complete, professional workflow.


## Flowchart of the Desired Analysis Outcome (Multiple Comparisons)
```mermaid
flowchart TD
    Start[Start: Load Data & EDA] --> Assumptions{Check Assumptions<br/>Normality per group + Homogeneity (Levene)}
    Assumptions -->|OK| Omnibus[One-way ANOVA<br/>(overall test for any difference)]
    Omnibus -->|Significant| PostHoc[Post-hoc Pairwise Tests<br/>with correction: Bonferroni or Tukey HSD]
    Omnibus -->|Not Significant| Stop[No further pairwise tests needed]<br/>[Report overall: no evidence of differences]
    PostHoc --> EffectSize[Compute Cohen's d + 95% CI for each significant pair]
    EffectSize --> Interpret[Interpret: Which stores differ?<br/>Practical significance + Business impact]
    Interpret --> Audience[Tailor Reporting to Audience<br/>Execs: 'Store B outperforms A'<br/>Analysts: Full stats, corrections, limitations]
    Audience --> Conclusion[Conclusion & Recommendations<br/>Follow data analysis report structure]
    Conclusion --> Simulation[Monte Carlo Simulation<br/>Modify n, effect sizes, correction → see FWER & power]
    Simulation --> End[End]
    Assumptions -->|Violated| Robust[Consider Welch ANOVA / Kruskal-Wallis<br/>or data transformation]
    Robust --> PostHoc
```
**Note:** This flowchart shows the proper modern workflow for comparing >2 groups (avoiding inflated Type I error from naive pairwise t-tests). Include it in your reports.


## 1. Setup, Data Loading & EDA

**Instructions:**
1. Import `pandas`, `numpy`, `matplotlib.pyplot`, `seaborn`, `scipy.stats`, and `statsmodels.stats.multicomp`
2. Load `veryants.csv` into a DataFrame called `veryants`
3. Create three Series/vectors: `a`, `b`, `c` for stores A, B, C
4. Print value counts, means, and standard deviations per store
5. Create a nice boxplot (or boxenplot/violin) using seaborn showing Sales by Store. Add title and labels.
6. In a markdown cell: From the plot, which store(s) appear to have higher/lower average sales? Any obvious differences in spread?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.stats.multicomp as mc   # for Tukey HSD

# TODO: Load data
veryants = pd.read_csv('veryants.csv')

# TODO: Create group vectors
a = veryants.Sale[veryants.Store == 'A']
b = veryants.Sale[veryants.Store == 'B']
c = veryants.Sale[veryants.Store == 'C']

print('Observations per store:')
print(veryants['Store'].value_counts())

# TODO: Summary stats
print('\nMean sales by store:')
print(veryants.groupby('Store')['Sale'].mean())

# TODO: Visualization
plt.figure(figsize=(8,5))
sns.boxplot(data=veryants, x='Store', y='Sale', palette='Set2')
plt.title('Sales Distribution by VeryAnts Store Location')
plt.ylabel('Sale amount (USD)')
plt.xlabel('Store')
plt.show()


## 2. Assumption Checks

**Instructions:**
1. For each group, create Q-Q plots and run `shapiro.test` (or `stats.shapiro`).
2. Run Levene’s test for homogeneity of variances across all three groups: `stats.levene(a, b, c)`
3. In markdown: Are assumptions reasonably met for ANOVA / t-tests? What would you do if variances were very different?


In [ ]:
# TODO: Q-Q plots (one per group or in subplots)
fig, axes = plt.subplots(1, 3, figsize=(12,4))
for i, (group, name) in enumerate([(a,'A'), (b,'B'), (c,'C')]):
    stats.probplot(group, dist='norm', plot=axes[i])
    axes[i].set_title(f'Q-Q Plot Store {name}')
plt.tight_layout()
plt.show()

# TODO: Shapiro-Wilk per group
for group, name in [(a,'A'), (b,'B'), (c,'C')]:
    print(f'Shapiro-Wilk Store {name}: p = {stats.shapiro(group).pvalue:.4f}')

# TODO: Levene's test
lev = stats.levene(a, b, c)
print(f'\nLevene test for equal variances: statistic={lev.statistic:.3f}, p={lev.pvalue:.4f}')


## 3. Omnibus Test: One-way ANOVA

**Instructions:**
1. Run `stats.f_oneway(a, b, c)` and print the F-statistic and p-value.
2. Decide at α=0.05 whether there is evidence of *any* difference in mean sales across the three stores.
3. Why is it better to start with ANOVA rather than jumping straight to three pairwise t-tests?


In [ ]:
# TODO: One-way ANOVA
f_stat, p_anova = stats.f_oneway(a, b, c)
print(f'One-way ANOVA: F = {f_stat:.3f}, p = {p_anova:.5f}')

alpha = 0.05
print(f'Significant overall difference at α={alpha}? {p_anova < alpha}')


## 4. Post-hoc Pairwise Comparisons (with Correction)

**Instructions:**
1. Run the three pairwise t-tests (you can use `ttest_ind` with `equal_var=True` or `False`).
2. Print the three raw p-values.
3. Apply a simple Bonferroni correction manually (multiply each p by 3, cap at 1.0) or use `statsmodels.stats.multicomp`.
4. (Advanced) Run Tukey HSD using `mc.pairwise_tukeyhsd` — this is often preferred as it controls family-wise error while being less conservative than Bonferroni.
5. Decide significance for each pair after correction.

**Key concept to understand:** Without correction, the probability of at least one false positive across 3 tests is ~1 - (0.95)^3 ≈ 14.3% (not 5%). This is the multiple testing problem.


In [ ]:
# TODO: Pairwise t-tests (raw p-values)
pairs = [('A','B', a, b), ('A','C', a, c), ('B','C', b, c)]
raw_pvals = {}
for name1, name2, g1, g2 in pairs:
    _, p = stats.ttest_ind(g1, g2, equal_var=True)
    raw_pvals[f'{name1} vs {name2}'] = p
    print(f'{name1} vs {name2} raw p = {p:.5f}')

# TODO: Bonferroni correction (manual)
print('\nBonferroni corrected p-values (p * 3):')
for pair, p in raw_pvals.items():
    bonf_p = min(p * 3, 1.0)
    print(f'{pair}: corrected p = {bonf_p:.5f}')

# TODO (Advanced): Tukey HSD
# tukey = mc.pairwise_tukeyhsd(veryants['Sale'], veryants['Store'])
# print(tukey.summary())


## 5. Effect Sizes & Confidence Intervals

**Instructions:**
For each pair that remains significant after correction, calculate Cohen’s d and the 95% CI for the mean difference (you can reuse the pooled SD formula or extract from t-test objects).
Interpret the practical size of the differences (small/medium/large).


In [ ]:
# TODO: Cohen's d and CI for key pairs (example for A vs B)
# You can copy the pooled_sd + cohens_d formula from previous exercises
# or compute it inside a small function.
print('Calculate Cohen\'s d and 95% CI for significant pairs after correction.')


## 6. More Practice Exercises

1. Re-run the pairwise tests using Welch’s t-test (`equal_var=False`). Do conclusions change?
2. Try a different correction method (e.g., Holm-Bonferroni — `statsmodels` has `multipletests`).
3. Interpret the results from a business perspective: Which store should the company focus on? What might explain higher sales at B?
4. (Advanced) Use `statsmodels.stats.multicomp.pairwise_tukeyhsd` and interpret the `reject` column and `meandiff`.
5. What if the design was unbalanced (different n per store)? How would that affect your choice of test?


## 7. Simulation: Multiple Testing, Family-Wise Error Rate & Power

**Goal:** See how often you get false positives when there are *no* true differences, and how correction helps. Also explore power when real differences exist.

**Instructions:** Modify the parameters below (especially `true_means`, `n_per_group`, `n_simulations`, and whether you apply `correction`).
Observe the proportion of simulations where at least one pairwise test is significant (this is the family-wise error rate when all means are equal).


In [ ]:
np.random.seed(42)

# === MODIFIABLE PARAMETERS ===
true_means = [58.0, 65.0, 62.0]   # change to [60, 60, 60] for null (all equal)
sigma = 15.0
n_per_group = 150
n_simulations = 500
alpha = 0.05
apply_correction = True   # try False to see inflated FWER

# === Simulation skeleton ===
false_positives = 0

for i in range(n_simulations):
    g1 = np.random.normal(true_means[0], sigma, n_per_group)
    g2 = np.random.normal(true_means[1], sigma, n_per_group)
    g3 = np.random.normal(true_means[2], sigma, n_per_group)
    
    p12 = stats.ttest_ind(g1, g2, equal_var=True)[1]
    p13 = stats.ttest_ind(g1, g3, equal_var=True)[1]
    p23 = stats.ttest_ind(g2, g3, equal_var=True)[1]
    
    pvals = np.array([p12, p13, p23])
    
    if apply_correction:
        # Simple Bonferroni
        pvals = np.minimum(pvals * 3, 1.0)
    
    if np.any(pvals < alpha):
        false_positives += 1

fwer = false_positives / n_simulations
print(f'Family-wise error rate (at least one false positive): {fwer:.3f}')
print(f'When all means equal and no correction → expect ~0.14 (14%)')
print(f'With Bonferroni → should be close to or below {alpha}')


## 8. Conclusion & Audience-Aware Reporting

Write a short Conclusion section suitable for a data analysis report (see `paper-structure.pdf`).
Tailor it to different audiences using the guidance from the audience PDFs:
- **Executives**: Clear headline — which store(s) have significantly different (higher/lower) sales? Recommendation?
- **Technical supervisor**: Full stats (ANOVA F/p, which pairs significant after correction, effect sizes, assumptions, limitations of multiple testing).
- **Non-technical**: Simple language explaining why we used ANOVA + correction (“we checked overall first, then looked at specific pairs while being careful not to over-claim differences that could be due to chance”).

Include 1-2 sentences on practical takeaways: e.g., “Store B shows meaningfully higher average sales than Store A. Further investigation into location, assortment, or staff training at B could reveal best practices to replicate.”
